In [1]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset maestro
ruta_archivo = "../datasets/dataset_maestro_dashboard.csv"
df = pd.read_csv(ruta_archivo)
df['fecdoc'] = pd.to_datetime(df['fecdoc'])

# Ordenar cronologicamente para evitar errores en los rezagos
df = df.sort_values(by=['sucursal', 'producto', 'fecdoc']).reset_index(drop=True)

# 2. Ingenieria de Caracteristicas (Agrupado estricto por Sucursal y Producto)
print("Generando variables predictivas...")

# Rezagos temporales (Lags)
df['lag_1'] = df.groupby(['sucursal', 'producto'])['cantidad'].shift(1)
df['lag_7'] = df.groupby(['sucursal', 'producto'])['cantidad'].shift(7)
df['lag_14'] = df.groupby(['sucursal', 'producto'])['cantidad'].shift(14)
df['lag_30'] = df.groupby(['sucursal', 'producto'])['cantidad'].shift(30)

# Medias Moviles (Rolling Means)
df['rolling_mean_7'] = df.groupby(['sucursal', 'producto'])['cantidad'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
df['rolling_mean_30'] = df.groupby(['sucursal', 'producto'])['cantidad'].transform(lambda x: x.rolling(window=30, min_periods=1).mean())

# Variables de calendario
df['dia_semana'] = df['fecdoc'].dt.dayofweek
df['mes'] = df['fecdoc'].dt.month
df['es_fin_semana'] = df['dia_semana'].apply(lambda x: 1 if x >= 5 else 0)

# Eliminar nulos generados por los shifts iniciales
df = df.dropna().reset_index(drop=True)

# 3. Particion Temporal Estricta
fecha_corte = '2026-01-01'

df_train = df[df['fecdoc'] < fecha_corte].copy()
df_test = df[df['fecdoc'] >= fecha_corte].copy()

print("\nRESUMEN DE LA PARTICION TEMPORAL:")
print(f"Entrenamiento (2024-2025): {df_train['fecdoc'].min().date()} al {df_train['fecdoc'].max().date()} -> {len(df_train)} registros.")
print(f"Prueba (2026): {df_test['fecdoc'].min().date()} al {df_test['fecdoc'].max().date()} -> {len(df_test)} registros.")

# 4. Codificacion One-Hot para algoritmos de Machine Learning
columnas_categoricas = ['sucursal', 'producto']
df_train_encoded = pd.get_dummies(df_train, columns=columnas_categoricas, drop_first=True)
df_test_encoded = pd.get_dummies(df_test, columns=columnas_categoricas, drop_first=True)

# Alinear columnas para asegurar la misma estructura estructural
df_train_encoded, df_test_encoded = df_train_encoded.align(df_test_encoded, join='left', axis=1, fill_value=0)

# Definir variables predictoras (X) y objetivo (y)
columnas_a_excluir = ['fecdoc', 'cantidad', 'categoria_abc']
X_train = df_train_encoded.drop(columns=[col for col in columnas_a_excluir if col in df_train_encoded.columns])
y_train = df_train_encoded['cantidad']

X_test = df_test_encoded.drop(columns=[col for col in columnas_a_excluir if col in df_test_encoded.columns])
y_test = df_test_encoded['cantidad']

print("\nDataset procesado y listo para alimentar a los algoritmos.")

Generando variables predictivas...

RESUMEN DE LA PARTICION TEMPORAL:
Entrenamiento (2024-2025): 2024-01-06 al 2025-12-31 -> 33992 registros.
Prueba (2026): 2026-01-01 al 2026-07-30 -> 3781 registros.

Dataset procesado y listo para alimentar a los algoritmos.
